# Sensoriamento Remoto: Temperatura da Superfície e Imagens do Planetary Computer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/04_remote_sensing_lst_pc.pt.ipynb)

**Objetivo de aprendizagem**: baixar uma série temporal de Temperatura da Superfície (LST, *Land Surface Temperature*) do Microsoft Planetary Computer e, separadamente, obter imagens multiespectrais (Sentinel-2) do mesmo catálogo Planetary Computer — ambas recortadas ao contorno (footprint) de um mapa LCZ, de modo que se alinhem pixel a pixel com as classes LCZ calculadas nos notebooks anteriores.

## Por que Temperatura da Superfície, e por que recortar ao footprint LCZ primeiro

A Temperatura da Superfície (LST) é a principal variável observável por satélite usada como *proxy* para estudos de **Ilha de Calor Urbana de Superfície (SUHI)**. Diferente da temperatura do ar (medida na camada do dossel, a ~2 m, por estações meteorológicas — assunto da série `local/` mais adiante neste tutorial), a LST é a *temperatura radiométrica de pele* de tudo o que o sensor do satélite enxerga: telhados, ruas, copas de árvores, solo exposto, água. Ela responde quase instantaneamente ao aquecimento solar e às propriedades do material da superfície (albedo, emissividade, admitância térmica), o que a torna uma excelente lente para as diferenças no balanço de energia de superfície que o esquema de Zonas Climáticas Locais (LCZ) (Stewart & Oke, 2012) foi desenhado para capturar: uma LCZ 1 compacta de edifícios altos e uma LCZ 6 aberta e baixa absorvem e reirradiam energia solar de formas muito diferentes, e a LST torna essa diferença diretamente visível.

**Por que recortar ao footprint do mapa LCZ antes de qualquer outra coisa?** Toda análise posterior estratificada por LCZ — calcular a LST média por classe, rodar `lcz_uhi_surface` (intensidade da SUHI), ou usar a LST em um fluxo de índices espectrais — exige que o raster de LST e o raster de classificação LCZ estejam **exatamente na mesma grade**: mesma extensão, mesma resolução, mesmo sistema de referência de coordenadas, pixel a pixel. Se não estiverem espacialmente alinhados, qualquer estatística por classe mistura silenciosamente a LST de um local com o rótulo LCZ de outro. Tanto `lcz_get_lst` quanto `lcz_get_planetary_computer` recebem o GeoTIFF do mapa LCZ como grade de referência (`x=`) e reprojetam/reamostram tudo o que baixam sobre essa grade antes de retornar qualquer coisa — assim o problema de alinhamento é resolvido uma única vez, na ingestão, em vez de repetido a cada etapa posterior.

**Duas fontes de LST, dois compromissos bem diferentes:**

- **GOES-R ABI-L2-LST** (`source="goes"`) é um produto de satélite geoestacionário que cobre apenas as Américas (centrado nos EUA/CONUS), com resolução nativa grosseira de ~2 km — mas escaneia o mesmo disco a cada ~10-15 minutos, de modo que um único dia pode ser amostrado em quase qualquer horário solar local (`target_hour=`). Essa densidade temporal é valiosa para a dinâmica diurna da SUHI (por exemplo, como a ilha de calor atinge o pico à tarde e se inverte à noite), mas o tamanho grosseiro do pixel e a cobertura restrita às Américas limitam seu uso para análises globais ou em escala de bairro.
- **Sentinel-3 SLSTR LST** (`source="sentinel3"`) é um sensor de órbita polar com **cobertura global** em resolução mais fina, ~1 km, mas com apenas 1-2 passagens por dia em horários locais variáveis — amostragem temporal mais grosseira em troca de funcionar em qualquer lugar do planeta. É essa a fonte usada abaixo, justamente por funcionar para qualquer cidade, não apenas para as que estão dentro da área de visão do GOES.

**Por que o acesso ao Planetary Computer para Sentinel-2/Landsat importa em seguida**: a LST sozinha nos diz *o quão quente* está uma superfície, mas não *por quê* — é por causa da cobertura vegetal, da densidade construída, da umidade do solo ou do albedo da superfície? Responder isso exige bandas de reflectância multiespectral (vermelho, NIR, SWIR, ...), que é exatamente o que `lcz_get_planetary_computer` obtém do Sentinel-2 ou Landsat. Essas bandas alimentam diretamente índices espectrais (NDVI, NDBI, NDWI, entre outros) — assunto do próximo notebook desta série — que por sua vez explicam *por que* uma classe LCZ é mais quente ou mais fria. Este notebook, portanto, fica na dobradiça entre "onde está quente" (LST) e "o que está causando isso" (índices espectrais a partir de imagens ópticas).

In [ ]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"

## 1 — Baixar um mapa LCZ de referência

Usamos **Vaduz** (Liechtenstein) como cidade de referência ao longo de todo o tutorial — uma caixa delimitadora minúscula mantém todos os rasters pequenos, então tanto o download de LST quanto o do Planetary Computer abaixo permanecem rápidos. `lcz_get_map` retorna o caminho para um GeoTIFF LCZ recortado; esse caminho é o argumento `x=` que toda função abaixo usa como grade de referência.

In [ ]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Vaduz")
print("mapa LCZ:", map_path)

## 2 — `lcz_get_lst`: uma série temporal de LST do Sentinel-3

`lcz_get_lst(x, source, start_date, end_date, ...)` baixa uma **pilha diária de LST** — uma banda por data, `(n_dias, H, W)` — recortada e reamostrada sobre a grade do mapa LCZ. Parâmetros-chave:

- `source`: `"goes"` (Américas, ~2 km, alta frequência de revisita), `"sentinel3"` (global, ~1 km, 1-2 passagens/dia), ou `"both"` (GOES primeiro, Sentinel-3 como reserva por dia).
- `start_date`/`end_date`: datas ISO, inclusivas. Usamos aqui uma janela curta de 5 dias para manter o download rápido — o Sentinel-3 revisita com frequência suficiente para que um punhado de dias normalmente produza pelo menos uma cena utilizável por dia.
- `target_hour`: hora solar local desejada para a escolha do GOES (ignorado para o Sentinel-3, que retorna a(s) passagem(ns) disponível(is) para o dia).
- `units`: `"C"` (padrão) ou `"K"`. O cache em disco sempre armazena Kelvin, então alternar `units` entre chamadas nunca dispara um novo download.

Retorna um `LCZLSTResult` com `.array` (a pilha), `.dates` (data ISO por banda), `.units` e `.path` (o GeoTIFF do cache, útil para `lcz_plot_map` ou `lcz_uhi_surface` depois). Vaduz está bem fora da área de visão CONUS do GOES, então `source="sentinel3"` é a única opção aqui — o que também é o ponto central: é a fonte que funciona para *qualquer* cidade do planeta.

In [ ]:
from LCZ4py import lcz_get_lst

lst = lcz_get_lst(
    map_path,
    source="sentinel3",
    start_date="2024-06-01",
    end_date="2024-06-05",
    units="C",
    cache=True,
)

print("formato do array (dias, H, W):", lst.array.shape)
print("unidades:", lst.units)
print("datas:", lst.dates)
import numpy as np
for d, band in zip(lst.dates, lst.array):
    print(f"  {d}: media={np.nanmean(band):.2f} {lst.units}, px validos={int(np.isfinite(band).sum())}")

**Lendo a saída**: cada linha acima é a LST média de um dia sobre o footprint LCZ de Vaduz, mais uma contagem de pixels válidos. Um dia com poucos pixels válidos (ou nenhuma linha impressa — `lcz_get_lst` descarta silenciosamente dias sem cena utilizável) indica cobertura de nuvens ou nenhuma passagem sobre aquele pixel naquele dia; isso é normal para uma área pequena e uma janela curta, e é exatamente por isso que estudos de SUHI costumam compor semanas a meses em vez de confiar em uma única data. A distinção Kelvin vs. Celsius afeta apenas o array retornado — o cache em disco é sempre Kelvin, então rodar novamente com um `units=` diferente é instantâneo.

## 3 — Visualizando a pilha de LST com `lcz_plot_map` (`data_type="continuous"`)

`lcz_plot_map` foi apresentado no notebook 02 para mapas categóricos de classes LCZ. Ele também tem um **modo de raster contínuo** (`data_type="continuous"`, ou `"auto"`, que detecta automaticamente uma pilha de tipo float) exatamente para esse tipo de saída: um GeoTIFF multibanda em que cada banda é um campo numérico, e não um código LCZ. Passe `band=` para escolher qual data renderizar — por índice, ou casando com uma descrição de banda (uma data ISO, neste caso). O parâmetro `colorscale=` (padrão `"RdBu_r"`) controla a rampa de cor contínua; tanto uma rampa divergente quanto uma sequencial térmica funcionam bem para LST.

In [ ]:
from LCZ4py import lcz_plot_map

plot_lst = lcz_plot_map(
    lst.path,
    data_type="continuous",
    band=lst.dates[0],
    title="LST Sentinel-3 — Vaduz",
    colorscale="inferno",
)
plot_lst.fig.show()

**Interpretação**: o mapa renderiza os valores brutos em Kelvin do arquivo de cache (o título da barra de cores reflete a tag `units` gravada no GeoTIFF) como um mapa de calor contínuo sobre o footprint de Vaduz — sem legenda de classes LCZ, apenas uma rampa de cor com barra de cores. Cores mais quentes marcam pixels de fundo de vale / área construída; cores mais frias marcam terreno de maior altitude ou vegetado. Este é o equivalente visual, para uma única data, das médias diárias impressas na seção 2 — útil para identificar padrões espaciais (por exemplo, um vale de rio quente, uma encosta florestada fria) que um único número médio não consegue mostrar.

## 4 — `lcz_list_pc_assets`: descobrir as bandas disponíveis antes de baixar

O Microsoft Planetary Computer hospeda mais de 100 coleções STAC. `lcz_get_planetary_computer` traz atalhos organizados (`PC_COLLECTIONS`) para as mais comuns — `"sentinel-2-l2a"`, `"landsat"`, `"worldcover"`, `"biodiversity"`, `"elevation"`, `"surface-water"`, `"landcover-cci"` — cada uma com uma lista padrão de bandas sensata. Antes de baixar qualquer coisa (especialmente para uma coleção que *não* está nessa lista de atalhos), `lcz_list_pc_assets(collection, x)` consulta o primeiro item STAC correspondente sobre a caixa delimitadora do mapa LCZ e retorna `{chave_da_banda: titulo}` — assim você sabe exatamente quais nomes de banda passar em `assets=`.

In [ ]:
from LCZ4py import lcz_list_pc_assets

assets = lcz_list_pc_assets("sentinel-2-l2a", map_path)
for key, title in assets.items():
    print(f"{key:12s} {title}")

**Lendo a saída**: cada linha é uma banda do Sentinel-2 disponível para este local — por exemplo, `B04` (vermelho), `B03` (verde), `B02` (azul), `B08` (NIR), além de bandas SWIR, camadas de classificação de cena e máscaras de qualidade. `PC_COLLECTIONS` já define como padrão para `"sentinel-2-l2a"` `["B04", "B03", "B02", "B08"]` (vermelho/verde/azul/NIR) — suficiente para um composto de cor verdadeira mais a banda NIR que os índices espectrais precisam — mas é essa listagem que você consultaria para adicionar bandas SWIR (por exemplo, `"B11"`) para índices como NDBI ou NDMI no próximo notebook.

## 5 — `lcz_get_planetary_computer`: imagens Sentinel-2 sobre o footprint LCZ

`lcz_get_planetary_computer(x, collection, assets=None, start_date=None, end_date=None, max_cloud_cover=30.0, max_items=10, ...)` busca no catálogo STAC a `collection` informada, filtra por cobertura de nuvens (para coleções ópticas que expõem essa propriedade) e mosaica até `max_items` cenas candidatas (melhor cobertura de nuvens primeiro, primeiro pixel válido vence) sobre a grade do mapa LCZ — o mesmo padrão de reprojetar-e-alinhar do `lcz_get_lst`. Parâmetros-chave:

- `collection`: um atalho de `PC_COLLECTIONS` (`"sentinel-2-l2a"` aqui) ou qualquer id de coleção bruta do Planetary Computer mais `assets=` explícito.
- `start_date`/`end_date`: obrigatórios juntos para coleções variáveis no tempo. Usamos uma janela de 3 meses (`max_items=3`) para dar ao filtro de cobertura de nuvens candidatas suficientes sem baixar dezenas de cenas.
- `max_cloud_cover`: percentual máximo de `eo:cloud_cover`, filtrado no lado do servidor.
- `max_items`: quantas cenas candidatas mosaicar por banda — mantido pequeno (2-3) aqui para permanecer rápido; uma área montanhosa e na fronteira de tiles como Vaduz pode precisar de mais de uma cena para cobrir totalmente o footprint.

Retorna um `LCZPCResult` com `.array` (`(n_bandas, H, W)`), `.bands` (nome da banda por fatia do array), `.item_ids` (quais cenas STAC contribuíram) e `.path` (GeoTIFF do cache).

In [ ]:
from LCZ4py import lcz_get_planetary_computer

s2 = lcz_get_planetary_computer(
    map_path,
    collection="sentinel-2-l2a",
    start_date="2024-06-01",
    end_date="2024-08-31",
    max_cloud_cover=20,
    max_items=3,
)

print("bandas:", s2.bands)
print("formato do array (bandas, H, W):", s2.array.shape)
print("itens STAC usados:", s2.item_ids)
import numpy as np
valid_px = int(np.isfinite(s2.array[0]).sum())
print("cobertura do footprint (banda 0, px validos):", valid_px, "/", s2.array[0].size)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# bandas sao ["B04", "B03", "B02", "B08"] = vermelho, verde, azul, NIR
def to_rgb(arr, band_idx=(0, 1, 2)):
    rgb = np.stack([arr[i] for i in band_idx], axis=-1)
    lo, hi = np.nanpercentile(rgb, [2, 98])
    rgb = np.clip((rgb - lo) / (hi - lo + 1e-9), 0, 1)
    return np.nan_to_num(rgb)

plt.figure(figsize=(6, 6))
plt.imshow(to_rgb(s2.array))
plt.title("Composicao em cor verdadeira Sentinel-2 L2A — Vaduz")
plt.axis("off")
plt.show()

**Interpretação**: a composição em cor verdadeira mostra o cenário alpino de vale de Vaduz — áreas construídas ao longo do fundo do vale, encostas florestadas e rocha exposta em altitudes maiores, tudo recortado exatamente ao mesmo footprint do mapa LCZ e da pilha de LST acima. `item_ids` revela se um único tile Sentinel-2 cobriu toda a área ou se vários foram mosaicados (comum perto de fronteiras de zona UTM ou de tile). Essa pilha de quatro bandas (`B04`/`B03`/`B02`/`B08`) é a entrada direta que `lcz_get_indices`, no próximo notebook, usa para calcular NDVI, NDBI e outros índices espectrais — a ponte entre "aqui está o dado de reflectância" e "aqui está o que ele nos diz sobre vegetação, densidade construída e umidade."

## Conclusão

Este notebook cobriu dois fluxos de dados de satélite complementares, ambos alinhados ao footprint de um mapa LCZ antes de qualquer análise: `lcz_get_lst` para Temperatura da Superfície (a principal variável observável de SUHI, negociando a alta frequência de revisita do GOES e sua cobertura restrita às Américas contra o alcance global e a revisita mais grosseira do Sentinel-3), e `lcz_get_planetary_computer` para acesso sem chave a imagens multiespectrais Sentinel-2/Landsat via o catálogo STAC da Microsoft, com `lcz_list_pc_assets` para descobrir os nomes das bandas primeiro. Recortar ambos à mesma grade LCZ garante que qualquer estatística posterior por classe (por exemplo, `lcz_uhi_surface`, `lcz_cal_indices`) compare o que é comparável, pixel a pixel.

**Anterior**: [`03_morphological_parameters`](03_morphological_parameters.pt.ipynb) — parâmetros morfológicos por pixel LCZ.

**Próximo**: [`05_spectral_indices`](05_spectral_indices.pt.ipynb) — transformar as bandas Sentinel-2 baixadas aqui em NDVI, NDBI e outros índices espectrais, e resumi-los por classe LCZ.